In [1]:
%%capture
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install qdrant-client langchain langchain-openai sentence-transformers
!pip install -q -U bitsandbytes transformers peft accelerate

In [2]:
%%capture
import pandas as pd
import requests
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from transformers import GenerationConfig, AutoTokenizer, AutoModelForCausalLM
import torch
import gc

In [3]:

model_name = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer1 = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Добавляем pad_token если его нет
if tokenizer1.pad_token is None:
    tokenizer1.pad_token = tokenizer1.eos_token

# Убираем BitsAndBytesConfig для CPU (квантизация не нужна на CPU)
# и явно указываем device_map="cpu"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

#model_name = 'Qwen/Qwen2.5-0.5B'
device, dtype = device, dtype


model1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map=device,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

model1.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [4]:
generation_config = GenerationConfig.from_pretrained(model_name)
generation_config.enable_thinking = False
generation_config.temperature = 0.3
generation_config.max_new_tokens = 500

In [5]:
def tell(model,tokenizer,string):
  input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': string}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
  input_ids = input_ids.to(model.device)

  with torch.no_grad():
      output = model.generate(
          input_ids,
          generation_config=generation_config
      )[:, input_ids.shape[-1]:].detach().cpu()

  output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
  return output

In [6]:
df=pd.read_json('https://huggingface.co/datasets/bearberry/rus_xquadqa/raw/main/rus_xquadqa.json')
print(df.columns.tolist())
pd.set_option('display.max_columns', None)
df['answers_string'] = df['answers'].apply(lambda x: ':::'.join(x) if isinstance(x, list) else x)
df['context_string'] = df['context'].apply( lambda x: ':::'.join([item['chunk'] for item in x if isinstance(item, dict) and item.get('is_relevant')]) )
print(df.shape)
df.head()

['id', 'question', 'answers', 'normalized_answers', 'context', 'metadata']
(1190, 8)


,id,question,answers,normalized_answers,context,metadata,answers_string,context_string
0,0_rusxquad,Сколько очков уступила защита Пэнтерс?,[308],[308],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}",308,"Защита Пэнтерс уступила всего 308 очков, заняв..."
1,1_rusxquad,Сколько мешков за карьеру было у Джареда Аллена?,[136],[136],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}",136,Линия Пэнтерс также представила ди-энда-ветера...
2,2_rusxquad,Сколько блокировок записал на свой счет Люк Ки...,[118],[118],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}",118,"Дэвис собрал 51⁄2 мешков, четыре вынужденных п..."
3,3_rusxquad,Сколько мячей перехватил Джош Норман?,"[4, четыре]","[четыре, 4]",[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}",4:::четыре,"Защита Пэнтерс уступила всего 308 очков, заняв..."
4,4_rusxquad,Кто больше записал на свой счет мешков в коман...,[Кейван Шорт],[кейван шорты],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}",Кейван Шорт,Дифенсив тэкл Пробоула Кейван Шорт лидирует в ...


In [7]:
df_=df.drop_duplicates(subset=['context_string','answers_string','question'])
df_.shape

(1186, 8)

In [8]:
df.loc[0,'context']

[{'chunk': 'Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.',
  'is_relevant': True},
 {'chunk': 'Дифенсив тэкл Пробоула Кейван Шорт лидирует в команде с 11 мешками, а также обеспечил три потери мяча и получил два.',
  'is_relevant': False},
 {'chunk': 'Нападающий Марио Эдисон добавил 61⁄2 мешков.',
  'is_relevant': False},
 {'chunk': 'Линия Пэнтерс также представила ди-энда-ветерана Джареда Аллена, пятикратного участника Пробоула, который был активным лидером по количеству мешков в карьере НФЛ в количестве 136, вместе с ди-эндом Кони Или, у которого было 5 мешков всего за 9 стартов.',
  'is_relevant': False},
 {'chunk': 'Позади них для участия в Пробоуле также были выбраны два из трех стартовых лайнбекеров Пэнтерс: Томас Дэвис и Люк Кикли.',
  'is_relevant': False},
 {'chunk': 'Дэвис собрал 51⁄2 мешков, четыре вынужденных потери мяча и четыре перехвата, в то время как Кикли л

In [9]:
df_=df.drop(columns=['context_string','answers_string'])
df_.head()

,id,question,answers,normalized_answers,context,metadata
0,0_rusxquad,Сколько очков уступила защита Пэнтерс?,[308],[308],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}"
1,1_rusxquad,Сколько мешков за карьеру было у Джареда Аллена?,[136],[136],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}"
2,2_rusxquad,Сколько блокировок записал на свой счет Люк Ки...,[118],[118],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}"
3,3_rusxquad,Сколько мячей перехватил Джош Норман?,"[4, четыре]","[четыре, 4]",[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}"
4,4_rusxquad,Кто больше записал на свой счет мешков в коман...,[Кейван Шорт],[кейван шорты],[{'chunk': 'Защита Пэнтерс уступила всего 308 ...,"{'tag': None, 'is_answerable': True}"


In [10]:
list_of_context=sum([[a['chunk'] for a in df.loc[i,'context']] for i in range(df.shape[0])],[])
list_of_context=[a for a in set(list_of_context)]
map_of_context={list_of_context[i]:i for i in range(len(list_of_context))}
list_of_question=[df.loc[i,'question'] for i in range(df.shape[0])]
list_of_question=[a for a in set(list_of_question)]
map_of_question={list_of_question[i]:i for i in range(len(list_of_question))}

In [11]:
len(list_of_context)

1237

In [12]:
def transform(context,question,answer):
  context_=[  map_of_context[a['chunk']]  for a in context if a['is_relevant'] ]
  #question_=map_of_question[question]
  return (question,context_,answer)

In [13]:
data=[transform(df_.loc[i,'context'],df_.loc[i,'question'],df_.loc[i,'answers']) for i in range(df_.shape[0])]

In [14]:
data[0]

('Сколько очков уступила защита Пэнтерс?', [777], ['308'])

In [15]:
%%capture
corpus = QdrantClient(":memory:")
corpus.create_collection(
    collection_name="my_docs",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

sent2vec = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
vectors = sent2vec.encode(list_of_context)
points = []
for i, (sentence, vector) in tqdm(enumerate(zip(list_of_context, vectors))):
    points.append(
        PointStruct(
            id=i,
            vector=vector.tolist(),
            payload={"text": sentence}
        )
    )
corpus.upsert(
    collection_name="my_docs",
    points=points
)

In [16]:
def answer(que):
    query_vector = sent2vec.encode([que])[0]
    results = corpus.query_points(
        collection_name="my_docs",
        query=query_vector.tolist(),
        limit=40
    )

    indices = [point.id for point in results.points]

    context=' '.join([list_of_context[idx] for idx in indices])
    prompt='Ответь на вопрос: ' + que + ' используя данные: '+ context+' думай быстро, дай короткий логичный ответ из 1-2 слов'
    return tell(model1,tokenizer1,prompt),context

In [17]:
for i in tqdm(range(0,100,10)):
  a,b=answer(data[i][0])
  print('-'*50,'\n',data[i][0],'\n',a,'\n',data[i][2],'\n',b)

 10%|█         | 1/10 [00:03<00:29,  3.28s/it]

-------------------------------------------------- 
 Сколько очков уступила защита Пэнтерс? 
 308 
 ['308'] 
 Второй по популярности в Каролине сэйфти Пробоула Курт Колеман, который был лидером команды с максимальным показателем в своей карьере – 7 перехватов, при этом также набрав 88 перехватов, и корнербэк Пробоула Джош Норман, который в течение сезона превратился в шатдаун-корнера и имел четыре перехвата, два из которых были завершены тачдауном. Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул. Линия Пэнтерс также представила ди-энда-ветерана Джареда Аллена, пятикратного участника Пробоула, который был активным лидером по количеству мешков в карьере НФЛ в количестве 136, вместе с ди-эндом Кони Или, у которого было 5 мешков всего за 9 стартов. Тем временем, нападение Денвера удерживалось за пределами зоны защиты в течение трех игр, но штрафной корнербэку Джошу Норману принес Б

 20%|██        | 2/10 [00:05<00:21,  2.75s/it]

-------------------------------------------------- 
 Сколько перехватов в сезоне 2015 года сделала защита Пэнтерс? 
 24 
 ['24'] 
 Второй по популярности в Каролине сэйфти Пробоула Курт Колеман, который был лидером команды с максимальным показателем в своей карьере – 7 перехватов, при этом также набрав 88 перехватов, и корнербэк Пробоула Джош Норман, который в течение сезона превратился в шатдаун-корнера и имел четыре перехвата, два из которых были завершены тачдауном. Линия Пэнтерс также представила ди-энда-ветерана Джареда Аллена, пятикратного участника Пробоула, который был активным лидером по количеству мешков в карьере НФЛ в количестве 136, вместе с ди-эндом Кони Или, у которого было 5 мешков всего за 9 стартов. Дифенсив тэкл Пробоула Кейван Шорт лидирует в команде с 11 мешками, а также обеспечил три потери мяча и получил два. Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоу

 30%|███       | 3/10 [00:07<00:15,  2.15s/it]

...

-------------------------------------------------- 
 Как фамилия игрока, который забрал мяч у Ньютона в конце четвертого периода? 
 Эдуардson 
 ['Миллер'] 
 В следующей игре Миллер забрал мяч у Ньютона, и после того, как несколько игроков бросились к нему, он сделал длинный отскок назад и передал мяч Уарду , который вернул его на пять ярдов к 4-ярдовой линии Пэнтерс. Хотя несколько игроков бросились в кучу, чтобы попытаться забрать его, Ньютон этого не сделал, и его недостаточная агрессивность впоследствии подверглась серьезной критике. Например, Исаак Ньютон в своей теории всемирного тяготения объединил силу, ответственную за падение объектов на поверхность Земли, с силой, ответственной за орбиты небесных тел. На реконструкции Джонса и соавт. Плейстоценовая эпоха (P). Один из ископаемых образцов, о котором впервые сообщалось в 1996 году, имел большой рот, предположительно окруженный складкой мускульной природы. Например, баскетбольный мяч, брошенный с земли, летит по параболе, посколь

 90%|█████████ | 9/10 [00:20<00:02,  2.42s/it]

-------------------------------------------------- 
 Когда Полония Уорсоу выиграла чемпионат страны до 2000 года? 
 1946年 
 ['1946'] 
 Их местные конкуренты, Полония Уорсоу, имеют значительно меньше болельщиков, но им удалось выиграть Чемпионат Экстракласса в 2000 году. Они также выиграли чемпионат страны в 1946 года, а также дважды выиграли кубок. В 2000 году он получил заслуженную награду лучшего университета года по версии Sunday Times. Плейстоценовая эпоха (P). Она была восстановлена в апреле 2000 года, после окончания послевоенного коммунистического контроля над страной и восстановления свободной рыночной экономики. Подпись из четырех нот с тех пор обновлялась каждый телевизионный сезон (хотя некоторые ее варианты, использовавшиеся с 1998 по 1999 сезон все еще используются при производстве карточек тщеславия, которые можно увидеть после заключительных титров большинства передач). С 1990-х годов к Тесле возродился общественный интерес. Затем они победили действующего чемпиона Супер

100%|██████████| 10/10 [00:22<00:00,  2.29s/it]

-------------------------------------------------- 
 Какой второй уровень территориального деления в Польше? 
 Уезды 
 ['повят', 'уезд', 'повет', 'уезды или поветы'] 
 Основной единицей территориального деления в Польше является коммуна (гмина). Такими городами являются, например, Люблин, Краков, Гданьск, Познань. В Варшаве ее округа также имеют некоторые права повета - например, уже упоминавшаяся регистрация автомобилей. Но, например, районы в Кракове не имеют прав повета, поэтому регистрационные номера в Кракове одного типа для всех округов. Сейчас они играют в 4-й лиге (5-й уровень в Польше) - нижней профессиональной лиге в структуре Национальной польской футбольной ассоциации (PZPN). На протяжении всего своего существования Варшава была мультикультурным городом. Некоторые более крупные города получают право, то есть задачи и привилегии, которыми обладают подразделения второго уровня территориального деления - уезды или поветы. С момента своего создания все больше государств-членов 